<a href="https://colab.research.google.com/github/aashrayaaryalcs24-star/AI_SEM5/blob/main/week_3_A_search_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# 8-PUZZLE A* SEARCH
import heapq

def print_state(state):
    for row in state:
        print(row)
    print()

def misplaced_tiles(state, goal):
    count = 0
    for i in range(3):
        for j in range(3):
            # Ignore blank
            if state[i][j] != 0:
                if state[i][j] != goal[i][j]:
                    count += 1
    return count

def manhattan_distance(state, goal):
    total = 0
    for i in range(3):
        for j in range(3):
            tile = state[i][j]
            # Ignore blank
            if tile != 0:
                for x in range(3):
                    for y in range(3):
                        if goal[x][y] == tile:
                            total += abs(i - x) + abs(j - y)
    return total

def generate_children_with_directions(state):
    children_with_dirs = []
    state_list = [list(row) for row in state]
    blank_row, blank_col = 0, 0
    for i in range(3):
        for j in range(3):
            if state_list[i][j] == 0:
                blank_row = i
                blank_col = j

    moves = {
        "UP": (-1, 0),
        "DOWN": (1, 0),
        "LEFT": (0, -1),
        "RIGHT": (0, 1)
    }

    for direction, (dr, dc) in moves.items():
        new_row = blank_row + dr
        new_col = blank_col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_state = [row[:] for row in state_list]
            new_state[blank_row][blank_col] = new_state[new_row][new_col]
            new_state[new_row][new_col] = 0
            new_state_tuple = tuple(tuple(row) for row in new_state)
            children_with_dirs.append((direction, new_state_tuple))
    return children_with_dirs

def generate_children(state):
    return [child for _, child in generate_children_with_directions(state)]

def A_STAR(start, goal, heuristic):
    OPEN = []
    CLOSED = set()
    g = {}
    parent = {}
    counter = 0
    all_solutions = []

    start = tuple(tuple(row) for row in start)
    goal = tuple(tuple(row) for row in goal)

    # Initial state
    g[start] = 0
    h = heuristic(start, goal)
    f = g[start] + h

    # Add start to OPEN
    heapq.heappush(OPEN, (f, counter, start))
    parent[start] = None
    counter += 1

    while OPEN:
        f_current, _, current = heapq.heappop(OPEN)

        if current in CLOSED:
            continue

        CLOSED.add(current)
        current_g = g[current]

        if current == goal:
            # Reconstruct this path
            path = []
            temp = current
            while temp is not None:
                path.append(temp)
                temp = parent[temp]
            path.reverse()
            all_solutions.append((f_current, path))
            continue

        for child in generate_children(current):
            if child in CLOSED:
                continue

            new_g = current_g + 1

            if child not in g or new_g < g[child]:
                g[child] = new_g
                parent[child] = current
                child_h = heuristic(child, goal)
                child_f = new_g + child_h
                heapq.heappush(OPEN, (child_f, counter, child))
                counter += 1

    if all_solutions:
        all_solutions.sort(key=lambda x: x[0])
        return all_solutions
    return None

start = (
    (2, 8, 3),
    (1, 6, 4),
    (7, 0, 5)
)

goal = (
    (1, 2, 3),
    (8, 0, 4),
    (7, 6, 5)
)

def display_path_with_all_directions(path, goal, heuristic):
    for i, state in enumerate(path):
        g_value = i
        h_value = heuristic(state, goal)
        f_value = g_value + h_value
        print(f"State {i} (Selected):")
        print_state(state)
        print(f"g = {g_value}, h = {h_value}, f = {f_value}")

        if i < len(path) - 1:
            print("--- Possible next moves from this state:")
            possible_moves = generate_children_with_directions(state)
            for direction, child_state in possible_moves:
                is_selected = "(CHOSEN)" if child_state == path[i+1] else ""
                child_g = g_value + 1
                child_h = heuristic(child_state, goal)
                child_f = child_g + child_h
                print(f"  Move {direction} {is_selected} -> f: {child_f} (g: {child_g}, h: {child_h})")
                print_state(child_state)
        print("========================================\n")

print("CASE 1: MISPLACED TILES")
solutions1 = A_STAR(start, goal, misplaced_tiles)
if solutions1:
    print(f"Found {len(solutions1)} path(s) to goal.")
    best_f, best_path = solutions1[0]
    print(f"Optimal Solution Cost (Min f-value): {best_f}")
    print("Number of moves in optimal path:", len(best_path) - 1)
    print("\nDetailed Path with all directions shown below:")
    display_path_with_all_directions(best_path, goal, misplaced_tiles)
else:
    print("No solution found.")

print("\n")
print("CASE 2: MANHATTAN DISTANCE")
solutions2 = A_STAR(start, goal, manhattan_distance)
if solutions2:
    print(f"Found {len(solutions2)} path(s) to goal.")
    best_f, best_path = solutions2[0]
    print(f"Optimal Solution Cost (Min f-value): {best_f}")
    print("Number of moves in optimal path:", len(best_path) - 1)
    print("\nDetailed Path with all directions shown below:")
    display_path_with_all_directions(best_path, goal, manhattan_distance)
else:
    print("No solution found.")

CASE 1: MISPLACED TILES
Found 1 path(s) to goal.
Optimal Solution Cost (Min f-value): 5
Number of moves in optimal path: 5

Detailed Path with all directions shown below:
State 0 (Selected):
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)

g = 0, h = 4, f = 4
--- Possible next moves from this state:
  Move UP (CHOSEN) -> f: 4 (g: 1, h: 3)
(2, 8, 3)
(1, 0, 4)
(7, 6, 5)

  Move LEFT  -> f: 6 (g: 1, h: 5)
(2, 8, 3)
(1, 6, 4)
(0, 7, 5)

  Move RIGHT  -> f: 6 (g: 1, h: 5)
(2, 8, 3)
(1, 6, 4)
(7, 5, 0)


State 1 (Selected):
(2, 8, 3)
(1, 0, 4)
(7, 6, 5)

g = 1, h = 3, f = 4
--- Possible next moves from this state:
  Move UP (CHOSEN) -> f: 5 (g: 2, h: 3)
(2, 0, 3)
(1, 8, 4)
(7, 6, 5)

  Move DOWN  -> f: 6 (g: 2, h: 4)
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)

  Move LEFT  -> f: 5 (g: 2, h: 3)
(2, 8, 3)
(0, 1, 4)
(7, 6, 5)

  Move RIGHT  -> f: 6 (g: 2, h: 4)
(2, 8, 3)
(1, 4, 0)
(7, 6, 5)


State 2 (Selected):
(2, 0, 3)
(1, 8, 4)
(7, 6, 5)

g = 2, h = 3, f = 5
--- Possible next moves from this state:
  Move DOWN  -> f: 6 (